In [ ]:
"""
realtime_anny.py
================================================================================
Real-time multi-camera pose -> live Anny mesh animation.

Single-threaded by design (no threads / queues / async):

    read frames -> detect (MediaPipe) -> triangulate -> solve pose
                -> temporal smoothing -> skin (Anny) -> draw (Open3D)

Cameras are calibrated ONCE from the first frame pair; every frame after that
runs only the cheap path above.

Missing keypoints are handled gracefully:
  * A proximal->distal CASCADE decides which bones can be solved this frame.
    The moment a limb loses a keypoint, that bone and everything downstream of
    it stop being solved.
  * A small temporal SMOOTHER relaxes any un-solved bone toward its LOCAL rest
    pose (i.e. rest *relative to its current, possibly moving, parent*) instead
    of snapping to the global bind pose. The same smoother low-pass-filters
    jitter and eases bones back in when they are re-acquired.
"""

import sys
import math
import time

import numpy as np
import torch
import cv2
import roma
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from xfeat_c.xfeat import XFeat
from scipy.spatial.transform import Rotation as Rot
import open3d as o3d                      # the single rendering dependency

sys.path.insert(0, "/Users/saptarshiMT/Music/meshsense/anny/src")
import anny


# =============================================================================
# CONFIG
# =============================================================================
DEVICE = "cpu"
DTYPE = torch.float32

# VIDEOS = [
#     "/Users/saptarshiMT/Music/meshsense/camera0.avi",
#     "/Users/saptarshiMT/Music/meshsense/camera1.avi",
# ]

VIDEOS = [0,1]

# For live capture, pass device indices to run() instead, e.g. run([0, 1]).

WIDTH, HEIGHT = 640, 480
MEDIAPIPE_MODEL = "/Users/saptarshiMT/Downloads/pose_landmarker_heavy.task"

# ONE phenotype, used for BOTH solving and display. (If these differ, the posed
# skeleton and the drawn body disagree -- which was a latent bug in the old code.)
PHENOTYPE_KWARGS = {
    "gender": 1, "age": 0.5, "height": 0.6, "weight": 0.1, "muscle": 0.1,
}

# Keypoints below this MediaPipe visibility are treated as "not detected".
MIN_VISIBILITY = 0.5

# Temporal smoothing -- all in SECONDS, so behaviour is frame-rate independent.
TAU_ACQUIRE = 0.08    # time constant for following fresh, tracked targets (fast)
TAU_RELAX   = 0.30    # time constant for relaxing a lost bone to rest  (slow)
HOLD        = 0.15    # grace period before a lost bone starts relaxing at all

# MediaPipe-world -> Anny-world axis fix (a -90 deg rotation about X).
M = np.array([[1, 0, 0],
              [0, 0, 1],
              [0, -1, 0]], dtype=np.float64)


# =============================================================================
# CAMERA INTRINSICS
# =============================================================================

def get_camera_matrix(w, h):
    sx, sy = w / 1920, h / 1080
    return np.array([
        [1950.78 * sx, 0.0,          960.0 * sx],
        [0.0,          1950.78 * sy, 540.0 * sy],
        [0.0,          0.0,          1.0       ],
    ], dtype=np.float64)


# =============================================================================
# MEDIAPIPE
# =============================================================================

def create_pose_landmarker(model_path):
    opts = vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=model_path),
        running_mode=vision.RunningMode.IMAGE,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return vision.PoseLandmarker.create_from_options(opts)


def get_pose_points(image, landmarker, min_visibility=MIN_VISIBILITY):
    """Return {landmark_idx: (x, y)} for confidently-detected, in-frame points."""
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not result.pose_landmarks:
        return {}
    h, w = image.shape[:2]
    pts = {}
    for idx, lm in enumerate(result.pose_landmarks[0]):
        vis = float(lm.visibility) if hasattr(lm, "visibility") else 1.0
        if vis < min_visibility:                 # confidence folds into presence
            continue
        x, y = int(lm.x * w), int(lm.y * h)
        if 0 <= x < w and 0 <= y < h:
            pts[idx] = (x, y)
    return pts


# =============================================================================
# CAMERA POSE ESTIMATION (run once) + TRIANGULATION
# =============================================================================

def estimate_cameras(frames, K):
    """Estimate every camera pose relative to camera 0. Run once, on frame 0."""
    xfeat = XFeat()
    Rs = [np.eye(3)]
    ts = [np.zeros(3)]
    Ps = [K @ np.hstack([np.eye(3), np.zeros((3, 1))])]
    for i in range(1, len(frames)):
        kp0, kpi = xfeat.match_xfeat_star(frames[0], frames[i], top_k=10000)
        pts0 = np.asarray(kp0, dtype=np.float64)
        ptsi = np.asarray(kpi, dtype=np.float64)
        E, mask = cv2.findEssentialMat(pts0, ptsi, K, method=cv2.RANSAC,
                                       prob=0.999, threshold=2.0)
        _, R, t, _ = cv2.recoverPose(E, pts0, ptsi, K, mask=mask)
        Rs.append(R)
        ts.append(t.flatten())
        Ps.append(K @ np.hstack([R, t]))
    return Rs, ts, Ps


def dlt_triangulate_point(Ps, pts):
    A = []
    for P, (x, y) in zip(Ps, pts):
        A.append(x * P[2] - P[0])
        A.append(y * P[2] - P[1])
    _, _, Vt = np.linalg.svd(np.array(A))
    Xh = Vt[-1]
    return Xh[:3] / Xh[3]


def triangulate_landmarks(Ps, pts2d_list):
    all_ids = set()
    for pts in pts2d_list:
        all_ids |= set(pts.keys())
    points3d = {}
    for idx in all_ids:
        valid_Ps = [Ps[ci] for ci, pts in enumerate(pts2d_list) if idx in pts]
        valid_pts = [pts[idx] for pts in pts2d_list if idx in pts]
        if len(valid_Ps) >= 2:                   # need >= 2 views to triangulate
            points3d[idx] = dlt_triangulate_point(valid_Ps, valid_pts)
    return points3d


# =============================================================================
# ANNY MODEL HELPERS
# =============================================================================

def build_model():
    return anny.create_fullbody_model(
        all_phenotypes=True, local_changes=True,
        remove_unattached_vertices=True, triangulate_faces=True,
    ).to(device=DEVICE, dtype=DTYPE)


def run_rest_pose(model):
    with torch.no_grad():
        out = model(pose_parameters={}, phenotype_kwargs=PHENOTYPE_KWARGS)
    return out, out["bone_poses"][0]


def run_native(extra, model, pose_no_root):
    """Forward kinematics: global bone transforms given base + extra params."""
    p = {**pose_no_root, **extra}
    with torch.no_grad():
        bp = model(pose_parameters=p, phenotype_kwargs=PHENOTYPE_KWARGS)["bone_poses"][0]
    return bp.cpu().numpy()


def skin(model, pose_params):
    """Full forward pass -> vertices (the per-frame mesh deformation)."""
    with torch.no_grad():
        out = model(pose_parameters=pose_params, phenotype_kwargs=PHENOTYPE_KWARGS)
    return out["vertices"].squeeze(0).cpu().numpy()


# =============================================================================
# ROTATION HELPERS
# =============================================================================

def _rigid(R3x3, model):
    """np 3x3 rotation -> roma.Rigid pose param on the model's device/dtype."""
    t = torch.from_numpy(np.ascontiguousarray(R3x3)).to(device=model.device, dtype=model.dtype)
    return roma.Rigid(linear=t.unsqueeze(0), translation=None)


def slerp(Ra, Rb, alpha):
    """Shortest-path interpolation between two scipy Rotations (alpha in [0, 1])."""
    rotvec = (Ra.inv() * Rb).as_rotvec()         # scipy returns angle in [0, pi]
    return Ra * Rot.from_rotvec(alpha * rotvec)


def rotation_between_vectors(a, b):
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    v = np.cross(a, b)
    s, c = np.linalg.norm(v), float(np.dot(a, b))
    if s < 1e-8:
        if c > 0:
            return np.eye(3)
        perp = np.array([1., 0, 0]) if abs(a[0]) < 0.9 else np.array([0, 1., 0])
        axis = np.cross(a, perp); axis /= np.linalg.norm(axis)
        return Rot.from_rotvec(np.pi * axis).as_matrix()
    Kx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + Kx + Kx @ Kx * ((1 - c) / (s ** 2))


def signed_angle_about_axis(v_from, v_to, axis):
    axis = axis / np.linalg.norm(axis)
    a = v_from - np.dot(v_from, axis) * axis
    b = v_to - np.dot(v_to, axis) * axis
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-8 or nb < 1e-8:
        return 0.0
    a /= na; b /= nb
    ang = np.arccos(np.clip(np.dot(a, b), -1.0, 1.0))
    if np.dot(np.cross(a, b), axis) < 0:
        ang = -ang
    return ang


def split_about_axis(R_world, weights, limits_deg):
    """Spread one rotation across a chain (per-bone weights, with deg clamps)."""
    rv = Rot.from_matrix(R_world).as_rotvec()
    theta = np.linalg.norm(rv)
    if theta < 1e-8:
        return [np.eye(3) for _ in weights]
    axis = rv / theta
    angles, used = [], 0.0
    for w, lim in zip(weights[:-1], limits_deg[:-1]):
        ti = float(np.clip(w * theta, -np.radians(lim), np.radians(lim)))
        angles.append(ti); used += ti
    angles.append(theta - used)
    return [Rot.from_rotvec(t * axis).as_matrix() for t in angles]


def set_param(bone, R_share, params, label_to_idx, model, pose_no_root):
    """Apply a WORLD rotation R_share to `bone`, stored as a LOCAL pose param."""
    bp = run_native(params, model, pose_no_root)
    C = bp[label_to_idx[bone], :3, :3]
    P = C.T @ R_share @ C                         # world rotation -> bone-local frame
    params[bone] = _rigid(P, model)


# =============================================================================
# LIMB SPECS  (proximal -> distal; rebuilt each frame because targets change)
# =============================================================================

def make_seg(targets, a, b):
    """Unit direction a->b in Anny frame, or None if either endpoint is missing."""
    if a not in targets or b not in targets:
        return None
    v = M @ (targets[b] - targets[a])
    return v / np.linalg.norm(v)


def make_arm_spec(targets):
    seg = lambda a, b: make_seg(targets, a, b)
    return {
        "L": [
            dict(chain=["clavicle.L", "shoulder01.L", "upperarm01.L"],
                 base="upperarm01.L", tip="lowerarm01.L",
                 target=seg(11, 13), weights=[0.15, 0.25, 0.60], limits=[20, 40, 180]),
            dict(chain=["lowerarm01.L"],
                 base="lowerarm01.L", tip="wrist.L",
                 target=seg(13, 15), weights=[1.0], limits=[180]),
            dict(chain=["wrist.L"],
                 base="wrist.L", tip="finger2-1.L",
                 target=seg(15, 19), weights=[1.0], limits=[180]),   # index finger
        ],
        "R": [
            dict(chain=["clavicle.R", "shoulder01.R", "upperarm01.R"],
                 base="upperarm01.R", tip="lowerarm01.R",
                 target=seg(12, 14), weights=[0.15, 0.25, 0.60], limits=[20, 40, 180]),
            dict(chain=["lowerarm01.R"],
                 base="lowerarm01.R", tip="wrist.R",
                 target=seg(14, 16), weights=[1.0], limits=[180]),
            dict(chain=["wrist.R"],
                 base="wrist.R", tip="finger2-1.R",
                 target=seg(16, 20), weights=[1.0], limits=[180]),
        ],
    }


def make_leg_spec(targets):
    seg = lambda a, b: make_seg(targets, a, b)
    return {
        "L": [
            dict(chain=["upperleg01.L"], base="upperleg01.L", tip="lowerleg01.L",
                 target=seg(23, 25), weights=[1.0], limits=[180]),
            dict(chain=["lowerleg01.L"], base="lowerleg01.L", tip="foot.L",
                 target=seg(25, 27), weights=[1.0], limits=[180]),
            dict(chain=["foot.L"], base="foot.L", tip="toe2-1.L",
                 target=seg(27, 31), weights=[1.0], limits=[180]),
        ],
        "R": [
            dict(chain=["upperleg01.R"], base="upperleg01.R", tip="lowerleg01.R",
                 target=seg(24, 26), weights=[1.0], limits=[180]),
            dict(chain=["lowerleg01.R"], base="lowerleg01.R", tip="foot.R",
                 target=seg(26, 28), weights=[1.0], limits=[180]),
            dict(chain=["foot.R"], base="foot.R", tip="toe2-1.R",
                 target=seg(28, 32), weights=[1.0], limits=[180]),
        ],
    }


# =============================================================================
# SOLVE POSE  (one frame -> desired LOCAL bone rotations, with cascade gating)
# =============================================================================

def _solve_limb(spec_side, desired, label_to_idx, model, pose_no_root):
    """
    Greedy proximal->distal IK for one limb.

    Iterates segments in order and STOPS at the first missing target
    (`break`) -- that is the cascade: every bone downstream of a missing
    keypoint is simply never written to `desired`, so the smoother relaxes
    it toward local rest under whatever parent we *did* manage to solve.
    """
    params = {}
    for seg in spec_side:
        if seg["target"] is None:
            break                                # <- cascade stop
        bp = run_native(params, model, pose_no_root)
        base = bp[label_to_idx[seg["base"]], :3, 3]
        tip = bp[label_to_idx[seg["tip"]], :3, 3]
        cur = tip - base
        cur /= np.linalg.norm(cur)
        R_world = rotation_between_vectors(cur, seg["target"])
        shares = split_about_axis(R_world, seg["weights"], seg["limits"])
        for bone, R_share in zip(seg["chain"], shares):
            set_param(bone, R_share, params, label_to_idx, model, pose_no_root)
            desired[bone] = params[bone].linear[0].cpu().numpy().astype(np.float64)


def solve_pose(targets, rest_out, rest_bone_poses, label_to_idx, model):
    """
    Map triangulated keypoints -> {bone: desired LOCAL rotation (np 3x3)}.

    Only bones we can trust THIS frame are returned; everything else is left
    out so the smoother relaxes it. Gating cascade:

        no hips             -> {}            (whole body relaxes to rest)
        hips, no shoulders  -> pelvis only   (no spine, no arms)
        per limb            -> stop at the first missing keypoint
    """
    have = lambda i: i in targets
    desired = {}

    # ---- gate: hips, or the whole character relaxes ----
    if not (have(23) and have(24)):
        return desired

    # ---- pelvis yaw (heading) ----
    pose_no_root = {}
    pelvis_dir = targets[23] - targets[24]
    horiz = np.array([pelvis_dir[0], 0.0, pelvis_dir[2]])
    n = np.linalg.norm(horiz)
    if n > 1e-8:
        horiz /= n
        yaw = -np.arctan2(horiz[2], horiz[0])
        R_pelvis = Rot.from_rotvec(yaw * np.array([0, 0, 1.0])).as_matrix()
        desired["pelvis.L"] = R_pelvis
        desired["pelvis.R"] = R_pelvis
        pelvis_param = _rigid(R_pelvis, model)
        pose_no_root["pelvis.L"] = pelvis_param
        pose_no_root["pelvis.R"] = pelvis_param

    # ---- spine + arms: require BOTH shoulders ----
    if have(11) and have(12):
        # spine = bend (lean) + twist, spread evenly over 5 vertebrae
        target_spine = M @ (0.5 * (targets[11] + targets[12])
                            - 0.5 * (targets[23] + targets[24]))
        target_spine /= np.linalg.norm(target_spine)
        target_sh_line = M @ (targets[11] - targets[12])
        target_sh_line /= np.linalg.norm(target_sh_line)

        heads = rest_out["rest_bone_heads"][0].cpu().numpy()
        anny_base = heads[label_to_idx["spine05"]]
        anny_sh_mid = 0.5 * (heads[label_to_idx["shoulder01.L"]]
                             + heads[label_to_idx["shoulder01.R"]])
        cur_spine = anny_sh_mid - anny_base
        cur_spine /= np.linalg.norm(cur_spine)
        cur_sh_line = heads[label_to_idx["shoulder01.L"]] - heads[label_to_idx["shoulder01.R"]]
        cur_sh_line /= np.linalg.norm(cur_sh_line)

        R_bend = rotation_between_vectors(cur_spine, target_spine)
        sl_after_bend = R_bend @ cur_sh_line
        twist = signed_angle_about_axis(sl_after_bend, target_sh_line, target_spine)
        R_torso = Rot.from_rotvec(twist * target_spine).as_matrix() @ R_bend
        R_step = Rot.from_rotvec(Rot.from_matrix(R_torso).as_rotvec() / 5).as_matrix()

        for name in ("spine05", "spine04", "spine03", "spine02", "spine01"):
            Bi = rest_bone_poses[label_to_idx[name], :3, :3].cpu().numpy().astype(np.float64)
            Pi = Bi.T @ R_step @ Bi
            desired[name] = Pi
            pose_no_root[name] = _rigid(Pi, model)

        arm_spec = make_arm_spec(targets)
        _solve_limb(arm_spec["L"], desired, label_to_idx, model, pose_no_root)
        _solve_limb(arm_spec["R"], desired, label_to_idx, model, pose_no_root)

    # ---- legs: only need hips (already guaranteed); per-side cascade ----
    leg_spec = make_leg_spec(targets)
    _solve_limb(leg_spec["L"], desired, label_to_idx, model, pose_no_root)
    _solve_limb(leg_spec["R"], desired, label_to_idx, model, pose_no_root)

    return desired


# =============================================================================
# TEMPORAL SMOOTHING  (per-bone, in LOCAL rotation space)
# =============================================================================

class PoseSmoother:
    """
    Each frame, every bone is eased toward a target rotation:

        tracked this frame -> its freshly-solved local rotation  (fast, TAU_ACQUIRE)
        lost this frame    -> identity = LOCAL REST, after a HOLD (slow, TAU_RELAX)

    Blending is shortest-path slerp in LOCAL param space. Because the param is
    local, a relaxing child keeps riding its (possibly moving) parent and
    relaxes toward rest *relative to that parent* -- never the global bind pose.
    That is exactly the raised-arm / lost-wrist case: the forearm holds its
    elbow bend, then gently straightens while still hanging off the live upper
    arm. The very same mechanism low-passes jitter and eases bones back in when
    they reappear.
    """

    def __init__(self, model):
        self.model = model
        self.state = {}     # bone -> scipy Rotation (its current local param)
        self.lost = {}      # bone -> seconds since it was last tracked

    def update(self, desired, dt):
        identity = Rot.identity()

        # 1) tracked bones: ease toward the fresh solution
        for bone, R in desired.items():
            target = Rot.from_matrix(R)
            if bone not in self.state:
                self.state[bone] = target           # first sighting: no lag
            else:
                a = 1.0 - math.exp(-dt / TAU_ACQUIRE)
                self.state[bone] = slerp(self.state[bone], target, a)
            self.lost[bone] = 0.0

        # 2) known-but-untracked bones: hold briefly, then relax to local rest
        for bone in self.state:
            if bone in desired:
                continue
            self.lost[bone] += dt
            if self.lost[bone] < HOLD:
                continue                            # hold: leave the pose alone
            a = 1.0 - math.exp(-dt / TAU_RELAX)
            self.state[bone] = slerp(self.state[bone], identity, a)

        return {bone: _rigid(R.as_matrix(), self.model)
                for bone, R in self.state.items()}


# =============================================================================
# REAL-TIME LOOP  (single thread: read -> solve -> smooth -> skin -> draw)
# =============================================================================

def read_frames(caps, size):
    frames = []
    for c in caps:
        ok, f = c.read()
        if not ok:
            return None
        frames.append(cv2.resize(f, size))
    return frames


def run(sources, size=(WIDTH, HEIGHT)):
    # --- setup ---
    model = build_model()
    label_to_idx = {n: i for i, n in enumerate(model.bone_labels)}
    rest_out, rest_bone_poses = run_rest_pose(model)
    faces = np.asarray(model.faces.cpu().numpy(), dtype=np.int32)
    K = get_camera_matrix(*size)
    smoother = PoseSmoother(model)

    # constant global orientation (upright / facing camera); never smoothed
    root_rigid = roma.Rigid(
        linear=roma.rotvec_to_rotmat(
            torch.tensor([[-math.pi / 2.5, 0.0, 0.0]], device=DEVICE, dtype=DTYPE)),
        translation=None)

    caps = [cv2.VideoCapture(s) for s in sources]
    if not all(c.isOpened() for c in caps):
        raise RuntimeError("Could not open one or more sources")
    landmarker = create_pose_landmarker(MEDIAPIPE_MODEL)

    # --- calibrate cameras ONCE from the first frame pair ---
    frames = read_frames(caps, size)
    if frames is None:
        raise RuntimeError("Could not read first frames")
    print("[calibrate] estimating camera poses ...")
    _, _, Ps = estimate_cameras(frames, K)
    print("[calibrate] done. streaming (close the window to stop) ...")


    cv2.namedWindow("Video Feed", cv2.WINDOW_NORMAL)
    fps_t = time.perf_counter()
    fps = 0.0


    # --- window + initial (rest) mesh ---
    V0 = skin(model, {"root": root_rigid})
    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(np.asarray(V0, dtype=np.float64))
    mesh.triangles = o3d.utility.Vector3iVector(faces)
    mesh.compute_vertex_normals()
    mesh.paint_uniform_color([0.55, 0.78, 0.82])

    vis = o3d.visualization.Visualizer()
    vis.create_window("Anny", width=960, height=720)
    vis.add_geometry(mesh)

    last_t = None
    try:
        while True:
            now = time.perf_counter()
            # real elapsed time -> frame-rate-independent smoothing.
            # (For frame-accurate playback of recorded video, use 1/video_fps.)
            dt = (1.0 / 30.0) if last_t is None else max(now - last_t, 1e-3)
            last_t = now

            # detect -> triangulate
            pts_list = [get_pose_points(f, landmarker) for f in frames]
            targets = triangulate_landmarks(Ps, pts_list)


            fps_now = time.perf_counter()
            fps = 1.0 / max(fps_now - fps_t, 1e-6)
            fps_t = fps_now

            preview = np.hstack(frames)
            cv2.putText(
                preview,
                f"FPS: {fps:.1f}",
                (12, 32),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 255, 0),
                2,
                cv2.LINE_AA,
            )
            cv2.imshow("Video Feed", preview)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break



            # solve -> smooth -> skin
            desired = solve_pose(targets, rest_out, rest_bone_poses, label_to_idx, model)
            pose_params = smoother.update(desired, dt)
            pose_params["root"] = root_rigid
            V = skin(model, pose_params)

            # draw: update VERTICES + normals only; topology is set once
            mesh.vertices = o3d.utility.Vector3dVector(np.asarray(V, dtype=np.float64))
            mesh.compute_vertex_normals()
            vis.update_geometry(mesh)
            if not vis.poll_events():             # window closed by user
                break
            vis.update_renderer()

            # advance to the next frame set
            frames = read_frames(caps, size)
            if frames is None:                    # end of video / camera lost
                break
    finally:
        landmarker.close()
        for c in caps:
            c.release()
        vis.destroy_window()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    run(VIDEOS)
    # Live capture (two webcams):  run([0, 1])

/Users/saptarshiMT/Music/meshsense/anny/src/anny/models/rigged_model.py:117: UserWarning: Fallback to default lbs skinning. Consider installing NVidia Warp for lower memory footprint.
  warnings.warn("Fallback to default lbs skinning. Consider installing NVidia Warp for lower memory footprint.")
I0000 00:00:1780558707.160672 2579619 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 19, prefix = pthread-default
I0000 00:00:1780558707.185893 2579619 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1780558707.230937 2579623 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780558707.244524 2579634 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[calibrate] estimating camera poses ...
loading weights from: /Users/saptarshiMT/Music/meshsense/xfeat_c/xfeat.pt
[calibrate] done. streaming (close the window to stop) ...


W0000 00:00:1780558707.977755 2579622 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
